# The cubic hidden in the residuals

The quadratic residuals $1, -2, 0, 2, -1$ are orthogonal to $1$, $x$, and $x^2$. They are not orthogonal to $x^3$. These five points lie exactly on a cubic, and the algebra identifies every coefficient.


## The odd block moves, the even block does not

Degree $3$ adds the column $x^3 = -8, -1, 0, 1, 8$. Its power sums against the even columns are zero:

$$
\sum x^3 = 0, \qquad \sum x^5 = -32 - 1 + 0 + 1 + 32 = 0.
$$

So $b_0$ and $b_2$ stay at the quadratic solution $1$ and $3$. The new unknowns are $b_1$ and $b_3$. With $\sum x^4 = 34$, $\sum x^6 = 64 + 1 + 0 + 1 + 64 = 130$, and

$$
\sum x^3 y = (-8)(10) + (1)(8) + (8)(16) = 56,
$$

the odd block is

$$
\begin{aligned}
10b_1 + 34b_3 &= 20, \\
34b_1 + 130b_3 &= 56.
\end{aligned}
$$

Divide by $2$:

$$
\begin{aligned}
5b_1 + 17b_3 &= 10, \\
17b_1 + 65b_3 &= 28.
\end{aligned}
$$

Five times the second equation, minus seventeen times the first:

$$
(325 - 289)b_3 = 140 - 170 \Longrightarrow 36b_3 = -30 \Longrightarrow b_3 = -\dfrac{5}{6}.
$$

Then

$$
5b_1 + 17\left(-\dfrac{5}{6}\right) = 10 \Longrightarrow 5b_1 = \dfrac{145}{6} \Longrightarrow b_1 = \dfrac{29}{6}.
$$

The cubic is

$$
1 + \dfrac{29}{6}x + 3x^2 - \dfrac{5}{6}x^3.
$$

The slope moved from $2 = 12/6$ to $29/6$ because $x$ and $x^3$ share the odd block. They are not orthogonal to each other. $\sum x\cdot x^3 = \sum x^4 = 34 \neq 0$.


## It passes through every point

$$
\begin{aligned}
x = -2 &\Longrightarrow 1 - \dfrac{29}{3} + 12 + \dfrac{20}{3} = 10, \\
x = -1 &\Longrightarrow 1 - \dfrac{29}{6} + 3 + \dfrac{5}{6} = 0, \\
x = 0 &\Longrightarrow 1, \\
x = 1 &\Longrightarrow 1 + \dfrac{29}{6} + 3 - \dfrac{5}{6} = 8, \\
x = 2 &\Longrightarrow 1 + \dfrac{29}{3} + 12 - \dfrac{20}{3} = 16.
\end{aligned}
$$

The residual sum of squares is $0$. A residual sum cannot be negative, and the four columns are linearly independent, so this coefficient vector is the unique least-squares cubic. One residual degree of freedom remains, $5 - 4 = 1$, and the residual on that degree of freedom happens to be zero. The variance estimate $s^2 = \mathrm{RSS}/(n-p)$ is zero. Every ordinary $t$ interval built from it has width zero. That is a true description of a table that satisfies a cubic equation exactly. It is not a precise estimate of a noise level, because the table supplied no leftover noise.


In [1]:
# The cubic coefficients, and a residual of zero at every sample point.
from fractions import Fraction
import numpy as np
x = [-2, -1, 0, 1, 2]
y = [10, 0, 1, 8, 16]
X = np.array([[1, xi, xi ** 2, xi ** 3] for xi in x], dtype=float)
beta = np.linalg.solve(X.T @ X, X.T @ np.array(y, dtype=float))
hand = [1, Fraction(29, 6), 3, Fraction(-5, 6)]
fitted = [1 + Fraction(29, 6) * xi + 3 * xi ** 2 - Fraction(5, 6) * xi ** 3 for xi in x]
print("beta", beta)
print("hand fitted", fitted)
assert np.allclose(beta, [float(value) for value in hand])
assert fitted == y
assert np.linalg.matrix_rank(X) == 4


beta [ 1.          4.83333333  3.         -0.83333333]
hand fitted [Fraction(10, 1), Fraction(0, 1), Fraction(1, 1), Fraction(8, 1), Fraction(16, 1)]


## One change: the quadratic residuals are the cubic's private column

The difference between the cubic and the quadratic is

$$
\left(\dfrac{29}{6} - 2\right)x - \dfrac{5}{6}x^3
= \dfrac{17}{6}x - \dfrac{5}{6}x^3.
$$

At the five inputs this difference equals $1, -2, 0, 2, -1$, which is exactly the quadratic residual. The quadratic had found the even part. Everything it called noise was the odd cubic term. There is no remainder after degree $3$.


In [2]:
# Cubic minus quadratic equals the quadratic residual at each sample point.
from fractions import Fraction
x = [-2, -1, 0, 1, 2]
gap = [Fraction(17, 6) * xi - Fraction(5, 6) * xi ** 3 for xi in x]
print("gap", gap)
assert gap == [1, -2, 0, 2, -1]


gap [Fraction(1, 1), Fraction(-2, 1), Fraction(0, 1), Fraction(2, 1), Fraction(-1, 1)]


## Pitfall

Do not divide by $s^2 = 0$ and announce an infinite $F$ for the cubic term as though a noise model had been confirmed. The drop in residual sum, from $10$ to $0$, is real. The ratio of that drop to a zero variance estimate is not a usable test. A polynomial that passes through every point has interpolated the table. The next degree, $x^4$, is then a square Vandermonde system: five coefficients and five distinct inputs, residual degrees of freedom zero, and the degree-$4$ coefficient on this particular table is $0$ because the cubic already fits.


In [3]:
# Degree 4 has a zero leading coefficient here, because the cubic already fits.
import numpy as np
x = np.array([-2.0, -1, 0, 1, 2])
y = np.array([10.0, 0, 1, 8, 16])
X = np.column_stack([x ** k for k in range(5)])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("degree-4 coefficients", beta)
assert np.allclose(beta[4], 0)
assert abs(np.sum((y - X @ beta) ** 2)) < 1e-8


degree-4 coefficients [ 1.00000000e+00  4.83333333e+00  3.00000000e+00 -8.33333333e-01
  3.23358125e-15]


## Summary

- The least-squares cubic is $1 + (29/6)x + 3x^2 - (5/6)x^3$. It passes through all five points.
- $b_0$ and $b_2$ are unchanged because $x^3$ is orthogonal to the even columns. $b_1$ moves from $2$ to $29/6$.
- $\mathrm{RSS} = 0$ with one degree of freedom left means $s^2 = 0$. The usual $t$ interval collapses.
